# AQUAVOLTA — Exploration and cleaning of detailed expenses

Sheet `DEPENSES` of `DATA BI.xlsx`.

**How to run** — install the dependencies once (`pip install -r analytics/requirements.txt`), place the confidential workbook in `data/raw/` (or set `AQUAVOLTA_DATA_DIR`), then run all cells from top to bottom. Each section reloads its own sheet into `res`, so sections can also be run independently after the two setup cells.

The source data is confidential: this notebook is published without outputs. See `analytics/README.md` and `analytics/calculation-audit.md`.

## Setup

In [ ]:
import pandas as pd
import missingno as msno
from sklearn.impute import SimpleImputer

In [ ]:
# Paths and loading helpers (see aquavolta_helpers.py)
from aquavolta_helpers import DATA_DIR, OUTPUT_DIR, load_sheet, save_output

print("Source workbooks :", DATA_DIR)
print("Cleaned outputs  :", OUTPUT_DIR)

## Sheet `DEPENSES`

Source: `DATA BI.xlsx` (header on row 3).

In [ ]:
res = load_sheet("DATA BI.xlsx", sheet_name='DEPENSES', header=2)
print(res)

In [ ]:
print(res["PAIEMENT"])

In [ ]:
print(res.shape)

In [ ]:
print(res.size)

In [ ]:
print(res.info())

In [ ]:
print(res.describe())

In [ ]:
print(res.columns)

In [ ]:
msno.matrix(res)

In [ ]:
msno.bar(res)

In [ ]:
msno.dendrogram(res)

In [ ]:
msno.heatmap(res)

In [ ]:
v = res.isnull().sum()
p = (v / len(res))*100
mis_table = pd.DataFrame({'Missing Values': v, 'Percentage': p})
print(mis_table)

In [ ]:
print(res.info())

In [ ]:
rows_missing_fct = res[res['N° FCT'].isna()]

print(rows_missing_fct)

In [ ]:
res = res.dropna(subset=['N° FCT'])
print(res.info())

In [ ]:
res["SODEXO"] = res["SODEXO"].fillna(0)
print(res)

In [ ]:
print(res.info())

In [ ]:
mode=res[["DESCRIPTION","NCT","PAIEMENT"]].mode().iloc[0]
for coll in mode.index:
    res[coll]=res[coll].fillna(mode[coll])
print(res.info())

In [ ]:
res['REMARQUE']=res['REMARQUE'].fillna("pas de REMARQUE")
print(res.info())

In [ ]:
mean = res['HT'].mean()
res['HT']=res['HT'].fillna(mean)
print(res.info())

In [ ]:
print(res[['HT', 'TVA', 'TTC']].head())

In [ ]:
print(res[['HT', 'TVA', 'TTC']].tail())

In [ ]:
res['tx_tva']=(res['TVA']/res['HT'])*100
print(res['tx_tva'])

In [ ]:
counts = res['tx_tva'].value_counts().sort_index()

print(counts)

In [ ]:
taux_legaux = [7, 13, 19]
counts_legaux = res[res['tx_tva'].isin(taux_legaux)]['tx_tva'].value_counts()

print(counts_legaux)

In [ ]:
pourcentages = (counts_legaux / counts_legaux.sum()) * 100
print(pourcentages)

> **Review note (audit finding F6).** `corriger_tva` assigns the 19 % rate to rows whose observed rate is `0` or missing, and to any rate outside the three tolerance bands. Expenses that are legitimately VAT-free therefore receive a computed VAT. Calculation left unchanged; to be confirmed with the business.

In [ ]:
def corriger_tva(t):
    if pd.isna(t) or t == 0:
        return 19
    elif 6 <= t <= 8:
        return 7
    elif 12 <= t <= 14:
        return 13
    elif 18 <= t <= 20:
        return 19
    else:
        return 19   
res['tx_tva'] = res['tx_tva'].apply(corriger_tva)
res['tx_tva'] = res['tx_tva'].fillna(19)
print(res.info())

In [ ]:
print(res['tx_tva'])

In [ ]:
res['TVA']=(res['HT']*res['tx_tva'])/100
print(res['TVA'])

In [ ]:
print(res.info())

In [ ]:
res = res.drop("tx_tva", axis=1)
print(res.info())

In [ ]:
print(res['TF'])

In [ ]:
imputer_mode = SimpleImputer(strategy='most_frequent')
res['TF'] = imputer_mode.fit_transform(res[['TF']])
print(res.info())

In [ ]:
res['TTC']=res['SODEXO']+res['HT']+res['TVA']+res['TF']
print(res.info())

In [ ]:
res["RS"] = res["RS"].fillna(0)
print(res.info())

In [ ]:
res["NET RS"]=res['TTC']-res['RS']
print(res.info())

In [ ]:
save_output(res, "OUTPUT_DEPENSE.xlsx")